# figure6.ipynb

Manuscript analysis and plotting notebook.

Run the configuration cell first. Input data and fonts are not bundled.
See the repository README and data requirements before executing.


In [ ]:
from pathlib import Path
import sys as _sys

_repo_root = next(
    (p for p in (Path.cwd(), *Path.cwd().parents)
     if (p / 'plot_config.py').is_file()
     and (p / 'config.example.json').is_file()),
    None,
)
if _repo_root is None:
    raise RuntimeError('Run this notebook from within the plotting repository.')
if str(_repo_root) not in _sys.path:
    _sys.path.insert(0, str(_repo_root))
from plot_config import configured_path


In [ ]:
import numpy as np  
import matplotlib.pyplot as plt  
import matplotlib.patches as patches 
import xarray as xr
import pandas as pd
import cartopy.crs as ccrs  
import cartopy.feature as cfeature
import cmaps
from cartopy.util import add_cyclic_point
import matplotlib.ticker as ticker
import matplotlib.dates as mdates

from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

In [ ]:
from scipy.stats import gaussian_kde
from scipy.ndimage import label

In [ ]:
fdir1 = configured_path('reccap_mask_file')
mask = xr.open_mfdataset(fdir1)
mask_ocean = mask.open_ocean

In [ ]:
fdir1 = configured_path('paired_mld_file')
ds = xr.open_mfdataset(fdir1)

time_new = pd.date_range(
    start='1982-01-1',
    periods=ds.sizes['time'],
    freq='MS'  # 每月开始
) + pd.Timedelta(days=14)  # 调整到每月中旬
ds = ds.assign_coords(time=time_new)
ds

In [ ]:
fdir3 = configured_path('de_boyer_climatology_file')
ds3 = xr.open_mfdataset(fdir3)
# 新建month坐标，覆盖原有time维度
month_vals = np.arange(1, 13)  # [1,2,...,12]
ds3 = ds3.assign_coords(month=("time", month_vals))
# 将维度名称从time重命名为month
ds3 = ds3.swap_dims({"time": "month"})
# 删除旧time坐标（可选，清理结构）
ds3 = ds3.drop_vars("time")

ds3["lon"] = xr.where(ds3.lon < 0, ds3.lon + 360, ds3.lon)
# 2. 把数据集按照新的lon从小到大重新排序，实现0.5~359.5
ds3 = ds3.sortby("lon")
ds3

In [ ]:
#map_true = mld_true.MLD_d003.where(mask_ocean)
map_true = ds3.mld_dr003.where(mask_ocean)
map_pred = ds.MLD_pred.where(mask_ocean).groupby('time.month').mean()
#map_pred2 = ds.MLD_pred.mean(dim=['time']).where(mask_ocean)

In [ ]:
diff = map_pred-map_true

In [ ]:
map_pred

# 纬向平均

In [ ]:
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
import string
from matplotlib.ticker import MultipleLocator

# ==============================
# 全局统一配色（所有洋盆共用）
# ==============================
COLOR_PRED = "#d62728"    # 预测统一蓝色
COLOR_TRUE = "#1f77b4"    # 观测统一红色

from pathlib import Path
from matplotlib import font_manager
font_dir = Path(configured_path('font_dir'))
font_files = [font_dir / 'Times New Roman.ttf', font_dir / 'Times New Roman Bold.ttf']
for font_file in font_files:
    if not font_file.is_file():
        raise FileNotFoundError(font_file)
    font_manager.fontManager.addfont(str(font_file))
resolved_font = Path(font_manager.findfont('Times New Roman', fallback_to_default=False)).resolve()
assert resolved_font in {font.resolve() for font in font_files}, resolved_font
plt.rcParams.update({
    'font.family': 'serif', 'font.serif': ['Times New Roman'],
    'mathtext.fontset': 'custom', 'mathtext.rm': 'Times New Roman',
    'mathtext.it': 'Times New Roman:italic', 'mathtext.bf': 'Times New Roman:bold',
    'axes.unicode_minus': False,
})
# ==============================
# 季节划分（气候态month维度）
# ==============================
winter_months = [12, 1, 2]
summer_months = [6, 7, 8]
season_info = [("DJF", winter_months), ("JJA", summer_months)]
season_names = [s[0] for s in season_info]
season_month_list = [s[1] for s in season_info]

# ==============================
# 三条经线（仅保留范围、名称，颜色不再单独分配）
# ==============================
meridian_info = [
    ("Pacific Section (180°", slice(170, 190)),
    ("Indian Section (75°E", slice(65, 85)),
    ("Atlantic Section (30°W", slice(320, 340))
]
n_meri = len(meridian_info)
n_seas = 2

# ==============================
# 创建画布 3行2列
# ==============================
fig, axes = plt.subplots(
    nrows=n_meri,
    ncols=n_seas,
    figsize=(14, 16),
    sharey="row"
)
letters = string.ascii_lowercase

# ==============================
# 循环绘图
# ==============================
for i_meri, (meri_name, lon_slc) in enumerate(meridian_info):
    for i_sea, (s_name, month_ids) in enumerate(zip(season_names, season_month_list)):
        ax = axes[i_meri, i_sea]

        # 1. 计算季节区域均值 + 经度维度标准差std
        pred_region = map_pred.sel(month=month_ids, lon=lon_slc).sortby("lat")
        true_region = map_true.sel(month=month_ids, lon=lon_slc).sortby("lat")

        # 均值（原有曲线）
        pred_mean = pred_region.mean(dim=["month", "lon"], skipna=True)
        true_mean = true_region.mean(dim=["month", "lon"], skipna=True)
        # 经度维度标准差：左右各5格点共10个lon格点的波动std
        pred_std = pred_region.std(dim="lon", skipna=True).mean(dim="month", skipna=True)
        true_std = true_region.std(dim="lon", skipna=True).mean(dim="month", skipna=True)
        
        # ========== 新增：沿纬度做5点滑动平滑 ==========
        window_size = 5
        pred_mean = pred_mean.rolling(lat=window_size, center=True, min_periods=1).mean()
        true_mean = true_mean.rolling(lat=window_size, center=True, min_periods=1).mean()
        pred_std = pred_std.rolling(lat=window_size, center=True, min_periods=1).mean()
        true_std = true_std.rolling(lat=window_size, center=True, min_periods=1).mean()

        # 印度洋截断至北纬30度
        if "Indian" in meri_name:
            pred_mean = pred_mean.where(pred_mean.lat <= 30, drop=True)
            true_mean = true_mean.where(true_mean.lat <= 30, drop=True)
            pred_std = pred_std.where(pred_std.lat <= 30, drop=True)
            true_std = true_std.where(true_std.lat <= 30, drop=True)

        lat = pred_mean.lat.values
        pred_mld = pred_mean.values
        true_mld = true_mean.values
        pred_err = pred_std.values
        true_err = true_std.values

        # ========== ±std阴影填充，全局统一颜色 ==========
        # 预测值 std阴影
        ax.fill_betweenx(
            lat,
            pred_mld - pred_err,
            pred_mld + pred_err,
            color=COLOR_PRED,
            alpha=0.1,
            zorder=1
        )
        # 观测真值 std阴影
        ax.fill_betweenx(
            lat,
            true_mld - true_err,
            true_mld + true_err,
            color=COLOR_TRUE,
            alpha=0.1,
            zorder=1
        )

        # 2. 绘图：全部实线，全局统一颜色区分pred/obs，不再分洋盆色
        ax.plot(pred_mld, lat, color=COLOR_PRED, linestyle="-", lw=3.5, label="SJTU-ViT", zorder=3)
        ax.plot(true_mld, lat, color=COLOR_TRUE, linestyle="-", lw=3.5, label="Obs (de Boyer)", zorder=3)

        # 只保留左下坐标轴
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)
        ax.tick_params(top=False, right=False, labelsize=24)
        
        # 子图标号 (a)(b)(c)...
        letter_idx = i_meri * n_seas + i_sea
        
        if letter_idx in [0, 4]:  # 0=a,4=e
            ax.yaxis.set_major_locator(MultipleLocator(30))
        elif letter_idx == 2:     # 2=c
            ax.yaxis.set_major_locator(MultipleLocator(20))

        ax.text(
            0.02, 1.08, f"({letters[letter_idx]})",
            transform=ax.transAxes, fontsize=24, va="top",
            bbox=dict(facecolor="white", edgecolor="none", alpha=0.7)
        )

        ax.set_title(f"{meri_name} , {s_name})", fontsize=24)
        ax.grid(alpha=0.2, linestyle="--", zorder=0)
        
        # Y轴标签 & 刻度文字：仅第一列(i_sea==0)显示
        ax.set_ylabel("Latitude", fontsize=23.5)
        if i_sea != 0:
            ax.set_ylabel("")
            ax.tick_params(labelleft=False)

        # X轴标签 & 刻度文字：仅最后一行(i_meri == n_meri - 1)显示
        ax.set_xlabel("MLD (m)", fontsize=23.5)
        if i_meri != n_meri - 1:
            ax.set_xlabel("")
            #ax.tick_params(labelbottom=False)
        # 仅每行第一张图显示y轴标签
        if i_sea != 0:
            ax.tick_params(labelleft=False)

        # 仅每行最右侧子图放图例
        if i_sea == 1:
            ax.legend(loc="upper right", fontsize=22)

plt.tight_layout()
output_dir = Path(configured_path('output_dir'))
output_dir.mkdir(parents=True, exist_ok=True)
fig.savefig(output_dir / 'figure6.png', dpi=600, bbox_inches='tight', pad_inches=0)
plt.show()